In [16]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# QUESTION-1)Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. E.g., prompt_text A_text. What is the exact character length (total number of string characters using Python's len() function, NOT the number of tokens) of the combined_text string for the row at index 51? Note: We follow zero-indexing here.

In [17]:
# load the datasets using hugging face datset library
from datasets import load_dataset

# here dataset in the form of dictionary
dataset = load_dataset('csv', data_files='/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

In [18]:
# extract train datset from dataset dictionary
my_dataset = dataset['train']

# create new column in train datset by applying function
def combined_text(data):
    data['combined_text'] = data['prompt'] + " " + data['A']
    return data

new_dataset = my_dataset.map(combined_text)

In [19]:
# extract row at 51 index means 52th row
row_51 = new_dataset[51]

char_length = len(row_51['combined_text'])
print(char_length)

614


# QUESTION2)Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer?  

In [20]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
print("Total Vocabulary Size :", tokenizer.vocab_size)


Total Vocabulary Size : 30522


# QUESTION3)Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token.  

In [21]:
print("token:",tokenizer.sep_token)
print("Separator token integer id:", tokenizer.sep_token_id)

token: [SEP]
Separator token integer id: 102


# QUESTION4)Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors). 

What is the exact geometric shape (dimensions) of the resulting input_ids tensor?

In [22]:
train_df=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

In [23]:
tokenized_inputs = tokenizer(
    train_df['prompt'].tolist(),
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

print(tokenized_inputs['input_ids'].shape)

torch.Size([2000, 128])


# Question-5)A standard bert-base-uncased model has a hidden embedding size of 768 dimensions and uses exactly 12 attention heads in each layer. 

In Transformer architecture, the hidden size is divided equally among the attention heads. What is the exact dimensionality (size) of each individual attention head?

In [24]:
hidden_size = 768
num_attention_heads = 12

head_size = hidden_size // num_attention_heads

print("Each individual attention head size:",head_size)

Each individual attention head size: 64


# Question6)Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object. 

What is the exact shape of the last_hidden_state tensor returned? 

Note: We follow zero-indexing here.

In [28]:
from transformers import AutoTokenizer, AutoModel

# load the model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

# extract row 0 and prompt column
text = train_df.loc[0,"prompt"]

tokenized_inputs = tokenizer(text ,return_tensors="pt") 

outputs = model(**tokenized_inputs)

print("Output last hidden state shape:",outputs.last_hidden_state.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Output last hidden state shape: torch.Size([1, 31, 768])


# Question7)Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token (which is always the token at index 0). What is the sum of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places).

In [29]:
# CLs token embedding at first sentence and first token 
cls_vector = outputs.last_hidden_state[0, 0, :]

# first 5 values
first5_value = cls_vector[:5]

result = first5_value.sum().item()
print("Sum of first five value:",round(result, 4))

Sum of first five value: -1.2001


# Question8)Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0). 

What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places

In [30]:

from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# load model with oputput attentions =True
model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [37]:
string = "Light-ion fusion is a technique."

# tokenize the string and return tensor
tokenized_inputs = tokenizer(string, return_tensors="pt")

# pass tokenized inputs into model
outputs = model(**tokenized_inputs,output_attentions=True)


# Question9)Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's .encode() method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the sentence_transformers.util.cos_sim() function. What is the resulting similarity score rounded to 4 decimal places? Note: We follow zero-indexing here.

In [42]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# extrain option b feature and prompt feature
prompt = train_df.loc[0,"prompt"]
option_b = train_df.loc[0,"B"]

embeddings = model.encode([prompt, option_b])

# cosine similarity between  prompt embedding and option b embedding
similarity = util.cos_sim(embeddings[0], embeddings[1])

score = similarity.item()
print(round(score, 4))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.7658


<!-- Question 11) -->

# Question11)

In [43]:
from datasets import load_dataset
from transformers import pipeline

# 1. Load dataset
dataset = load_dataset("csv", data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
train= dataset["train"]

# 2. Initialize zero-shot pipeline (defaults to facebook/bart-large-mnli)
classifier = pipeline("zero-shot-classification")

# 3. Get row 1 prompt
text = train[1]["prompt"]

# 4. Candidate labels
candidate_labels = [
    train[1]["A"],
    train[1]["B"],
    train[1]["C"]
]

# 5. Run classification
result = classifier(text, candidate_labels)

# 6. Extract top score
top_score = result["scores"][0]
print(round(top_score, 4))

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

0.4575


# Question12)

In [44]:
# Case 1: Softmax (default)
res_softmax = classifier(text, candidate_labels, multi_label=False)
sum_softmax = sum(res_softmax["scores"])
print(sum_softmax)

# Case 2: Sigmoid
res_sigmoid = classifier(text, candidate_labels, multi_label=True)
sum_sigmoid = sum(res_sigmoid["scores"])
print(sum_sigmoid)

# Difference
diff = abs(sum_softmax - sum_sigmoid)
print(round(diff, 4))

0.9999999701976776
0.0005096072964079212
0.9995
